In [0]:
"""Descarga la serie telemétrica de ayer para una estación específica usando la API da ANA."""

from __future__ import annotations

import json
import os
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Optional

import requests

USER_API_ANA = dbutils.widgets.get("USER_API_ANA")
PASS_API_ANA = dbutils.widgets.get("PASS_API_ANA")

BASE_URL = "https://www.ana.gov.br/hidrowebservice"
LOGIN_ENDPOINT = "/EstacoesTelemetricas/OAUth/v1"
SERIE_TELEMETRICA_ENDPOINT = "/EstacoesTelemetricas/HidroinfoanaSerieTelemetricaAdotada/v2"
DEFAULT_TIMEOUT = 60

STATION_CODE = 74100000  # Cambiar si se necesita otra estación
OUTPUT_DIR = Path("/Volumes/weather/raw/ana_volume/json/daily/")
INTERVALO_BUSCA = "DIAS_2"

SERIE_INTERVAL_CHOICES = ["DIAS_2"]


class AnaApiError(RuntimeError):
    """Errores específicos del webservice de la ANA."""


def _obtener_credenciales() -> tuple[str, str]:
    usuario = USER_API_ANA
    password = PASS_API_ANA
    if not usuario or not password:
        raise AnaApiError(
            "Configura las variables USER_API_ANA y PASS_API_ANA en el entorno o en un archivo .env"
        )
    return usuario, password


def log_api_ana(session: Optional[requests.Session] = None) -> str:
    usuario, password = _obtener_credenciales()
    sess = session or requests.Session()
    url = f"{BASE_URL}{LOGIN_ENDPOINT}"
    headers = {"Identificador": usuario, "Senha": password}
    response = sess.get(url, headers=headers, timeout=DEFAULT_TIMEOUT)
    try:
        response.raise_for_status()
        payload: dict[str, Any] = response.json()
    except (ValueError, requests.RequestException) as exc:
        raise AnaApiError(f"Error autenticando con ANA: {exc}") from exc

    try:
        token = payload["items"]["tokenautenticacao"]
    except (KeyError, TypeError) as exc:
        raise AnaApiError("La respuesta de login no contiene el token esperado") from exc

    if not isinstance(token, str) or not token:
        raise AnaApiError("Token de autenticación vacío o inválido")
    return token


def consultar_hidroinfoana_serie_telem_adotada(
    token: str,
    *,
    session: Optional[requests.Session] = None,
    codigos_estacoes: list[int],
    tipo_filtro_data: str = "DATA_LEITURA",
    data_busca: Optional[str] = None,
    intervalo: str = "DIAS_30",
) -> list[dict[str, Any]]:
    if not codigos_estacoes:
        raise AnaApiError("Debe informar al menos un código de estación")
    if len(codigos_estacoes) > 10:
        raise AnaApiError("El endpoint admite hasta 10 códigos de estación")
    if intervalo not in SERIE_INTERVAL_CHOICES:
        raise AnaApiError(f"Intervalo inválido. Use uno de: {', '.join(SERIE_INTERVAL_CHOICES)}")

    sess = session or requests.Session()
    url = f"{BASE_URL}{SERIE_TELEMETRICA_ENDPOINT}"
    params: dict[str, Any] = {
        "Codigos_Estacoes": ",".join(str(codigo) for codigo in codigos_estacoes),
        "Tipo Filtro Data": tipo_filtro_data,
        "Range Intervalo de busca": intervalo,
    }
    if data_busca:
        params["Data de Busca (yyyy-MM-dd)"] = data_busca

    headers = {"Authorization": f"Bearer {token}"}
    response = sess.get(url, headers=headers, params=params, timeout=DEFAULT_TIMEOUT)
    try:
        response.raise_for_status()
        payload: dict[str, Any] = response.json()
    except (ValueError, requests.RequestException) as exc:
        raise AnaApiError(
            f"Error consultando HidroinfoanaSerieTelemetricaAdotada: {exc}"
        ) from exc

    items = payload.get("items")
    if not isinstance(items, list):
        raise AnaApiError("La respuesta no contiene una lista de registros")
    return items


def _filter_by_date(registros: list[dict[str, Any]], target_date: date) -> list[dict[str, Any]]:
    objetivo = target_date.isoformat()
    filtrados: list[dict[str, Any]] = []
    for registro in registros:
        timestamp = registro.get("Data_Hora_Medicao", "")
        fecha = timestamp.split(" ")[0] if timestamp else ""
        if fecha == objetivo:
            filtrados.append(registro)
    return filtrados


def fetch_daily_level() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    target_date = date.today() - timedelta(days=1)
    filename = f"ANA_{target_date.strftime('%Y_%m_%d')}.json"
    filepath = OUTPUT_DIR / filename

    if filepath.exists():
        print(f"SKIP {filename}")
        return

    session = requests.Session()
    token = log_api_ana(session=session)
    registros = consultar_hidroinfoana_serie_telem_adotada(
        token,
        session=session,
        codigos_estacoes=[STATION_CODE],
        data_busca=target_date.isoformat(),
        intervalo=INTERVALO_BUSCA,
    )

    registros_yesterday = _filter_by_date(registros, target_date)
    datos_a_guardar = registros_yesterday or registros

    filepath.write_text(
        json.dumps(datos_a_guardar, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(
        f"OK {filename}",
        f"registros_guardados={len(datos_a_guardar)}",
    )


try:
    fetch_daily_level()
except AnaApiError as exc:
    raise SystemExit(f"✗ {exc}")